# Lab 3 — Exercise: Wrangling the Google Play Store Dataset

**ITCS 3162 — Introduction to Data Mining**

**Name:** James_Paek
**Date:** 05/27/2026

This dataset is *messier* than diamonds — it's scraped from the Google Play Store and has the kinds of problems you'll see in the wild: numbers stored as strings, units mixed with values, duplicates, weird placeholder strings, and missing values. That's the point: you'll do real cleaning.

**Source:** Kaggle's "Google Play Store Apps" dataset (mirrored on GitHub for stable URL access).

When you're done, **Restart & Run All**, download as `.ipynb`, and submit via Canvas.


## Setup

Run this cell. It loads the dataset directly from a GitHub URL.


In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/sumitgirwal/google-play-store-data-analysis/master/googleplaystore.csv"
df = pd.read_csv(URL)
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


## Exercise 1 — Initial inspection (10 pts)

In the cell below, print **all three**:
1. The shape of the DataFrame (rows, columns)
2. The output of `df.info()`
3. The output of `df.describe(include="all")`

Then in the markdown cell, answer the questions.


In [15]:
print("Shape:", df.shape)

df.info()

df.describe(include="all")

Shape: (10841, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  object 
 1   Category        10841 non-null  object 
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  object 
 4   Size            10841 non-null  object 
 5   Installs        10841 non-null  object 
 6   Type            10840 non-null  object 
 7   Price           10841 non-null  object 
 8   Content Rating  10840 non-null  object 
 9   Genres          10841 non-null  object 
 10  Last Updated    10841 non-null  object 
 11  Current Ver     10833 non-null  object 
 12  Android Ver     10838 non-null  object 
dtypes: float64(1), object(12)
memory usage: 1.1+ MB


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
count,10841,10841,9367.000000,10841,10841,10841,10840,10841,10840,10841,10841,10833,10838
unique,9660,34,NaN,6002,462,22,3,93,6,120,1378,2832,33
top,ROBLOX,FAMILY,NaN,0,Varies with device,"1,000,000+",Free,0,Everyone,Tools,"August 3, 2018",Varies with device,4.1 and up
freq,9,1972,NaN,596,1695,1579,10039,10040,8714,842,326,1459,2451
mean,NaN,NaN,4.193338,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,0.537431,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,4.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,4.300000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,4.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Answer the following:**

1. How many rows and columns does the dataset have?
2. Look at the `Reviews`, `Size`, `Installs`, and `Price` columns. What dtype does pandas assign them? Why is that a problem for a numeric column like `Reviews`?
3. Which columns appear to have missing values?

YOUR ANSWERS:
1. 10,841 rows and 13 columns
2. 'Reviews', 'Size', 'Installs', and 'Price' columns are stored as object types (string too). This is bad because numeric operations like averages, sorting, and calculations can't be performed accuratelty until the values are converted to numeric types.
3. The columns that have missing values include 'Rating', 'Type', 'Content Rating', 'Current Ver', and 'Android Ver'.


## Exercise 2 — Missing values (10 pts)

In the cell below, compute (a) the count of missing values per column, sorted descending, and (b) the **percentage** of missing values per column rounded to 2 decimal places. Then drop any row that is missing the `Rating` column (since it's the most important missingness to handle) and store the result in `df_r`.


In [16]:
# Counts of missing values per column (sorted descending)
missing_counts = df.isna().sum().sort_values(ascending=False)

print(missing_counts)

# Percentage missing per column (rounded to 2 decimal places)
missing_percent = ((df.isna().sum() / len(df)) * 100).round(2).sort_values(ascending=False)

print(missing_percent)

# Create df_r by dropping rows where Rating is NaN; print new shape
df_r = df.dropna(subset=["Rating"])

print("New shape:", df_r.shape)

Rating            1474
Current Ver          8
Android Ver          3
Content Rating       1
Type                 1
Size                 0
Reviews              0
Category             0
App                  0
Price                0
Installs             0
Last Updated         0
Genres               0
dtype: int64
Rating            13.60
Current Ver        0.07
Android Ver        0.03
Content Rating     0.01
Type               0.01
Size               0.00
Reviews            0.00
Category           0.00
App                0.00
Price              0.00
Installs           0.00
Last Updated       0.00
Genres             0.00
dtype: float64
New shape: (9367, 13)


## Exercise 3 — Duplicates (10 pts)

There are duplicate rows in this dataset (the same app appears multiple times). In the cell below:
1. Print how many fully-duplicate rows exist in `df_r`.
2. How many duplicate values are there in the `App` column specifically (apps appearing under different categories)?
3. Create `df_dedup` by dropping fully-duplicate rows and print its new shape.


In [17]:
# Fully duplicate rows
print("Fully duplicate rows:", df_r.duplicated().sum())

# Duplicate App names
print("Duplicate App names:", df_r["App"].duplicated().sum())

# Remove fully duplicate rows and print shape
df_dedup = df_r.drop_duplicates()

print("Shape after dedup:", df_dedup.shape)

Fully duplicate rows: 474
Duplicate App names: 1170
Shape after dedup: (8893, 13)


## Exercise 4 — Cleaning the `Installs` column (15 pts)

The `Installs` column looks like `"1,000,000+"` — a string with a `+` and commas. Convert it to an integer column. Steps:

1. Inspect a few unique values: `df_dedup["Installs"].unique()[:10]`
2. Remove `+` and `,` characters with `.str.replace()`
3. Convert to int with `.astype(int)` (or use `pd.to_numeric` with `errors="coerce"` if any values look weird)
4. Assign the cleaned values back to `df_dedup["Installs"]`
5. Verify with `df_dedup["Installs"].dtype` and `.head()`


In [18]:
# Inspect unique values
print(df_dedup["Installs"].unique()[:10])

# Clean Installs to integer
df_dedup["Installs"] = (
    df_dedup["Installs"]
    .astype(str)
    .str.replace("+", "", regex=False)
    .str.replace(",", "", regex=False)
)

df_dedup["Installs"] = pd.to_numeric(
    df_dedup["Installs"],
    errors="coerce"
)

# Drop rows where Installs could not convert
df_dedup = df_dedup.dropna(subset=["Installs"])

# Convert to integer
df_dedup["Installs"] = df_dedup["Installs"].astype(int)

# Verify
print(df_dedup["Installs"].dtype)

print(df_dedup["Installs"].head())

['10,000+' '500,000+' '5,000,000+' '50,000,000+' '100,000+' '50,000+'
 '1,000,000+' '10,000,000+' '5,000+' '100,000,000+']
int64
0       10000
1      500000
2     5000000
3    50000000
4      100000
Name: Installs, dtype: int64


/tmp/ipykernel_1947/1698461932.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dedup["Installs"] = (
/tmp/ipykernel_1947/1698461932.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dedup["Installs"] = pd.to_numeric(


## Exercise 5 — Cleaning the `Price` column (15 pts)

`Price` looks like `"0"` for free apps and `"$4.99"` for paid ones. Convert it to a float.

1. Use `.str.replace("$", "", regex=False)` then `.astype(float)`.
2. Watch for any unexpected values (the real dataset has at least one weird row — the column may have a non-numeric placeholder that `pd.to_numeric(..., errors="coerce")` handles gracefully).
3. After cleaning, print how many paid apps there are (where `Price > 0`).


In [19]:
# Clean Price to float
df_dedup["Price"] = (
    df_dedup["Price"]
    .astype(str)
    .str.replace("$", "", regex=False)
)

df_dedup["Price"] = pd.to_numeric(
    df_dedup["Price"],
    errors="coerce"
)

# Drop rows where Price could not convert
df_dedup = df_dedup.dropna(subset=["Price"])

# Count paid apps
print("Paid apps:", (df_dedup["Price"] > 0).sum())

Paid apps: 613


## Exercise 6 — Filtering and a derived column (15 pts)

1. Create `top_apps`, containing only apps with `Rating >= 4.5` and `Installs >= 1_000_000`.
2. Print its shape and the top 5 categories of those apps by count.
3. In `df_dedup`, add a new column `Revenue` defined as `Price * Installs` (zero for free apps).
4. Show the top 10 apps by `Revenue`.


In [20]:
# Top apps and category counts
top_apps = df_dedup[
    (df_dedup["Rating"] >= 4.5) &
    (df_dedup["Installs"] >= 1000000)
]

print("Top apps shape:", top_apps.shape)

print(top_apps["Category"].value_counts().head())

# Revenue column and top 10
df_dedup["Revenue"] = df_dedup["Price"] * df_dedup["Installs"]

top_10 = (
    df_dedup[["App", "Revenue"]]
    .sort_values(by="Revenue", ascending=False)
    .head(10)
)

print(top_10)

Top apps shape: (1234, 13)
Category
GAME                  272
FAMILY                171
TOOLS                  89
HEALTH_AND_FITNESS     82
PRODUCTIVITY           62
Name: count, dtype: int64
                                App     Revenue
4347                      Minecraft  69900000.0
2241                      Minecraft  69900000.0
5351                      I am rich  39999000.0
5356              I Am Rich Premium  19999500.0
4034                  Hitman Sniper   9900000.0
7417  Grand Theft Auto: San Andreas   6990000.0
2883            Facetune - For Free   5990000.0
5578        Sleep as Android Unlock   5990000.0
8804            DraStic DS Emulator   4990000.0
4367       I'm Rich - Trump Edition   4000000.0


## Exercise 7 — Group-by summary (15 pts)

Using `df_dedup`, produce a single summary table with one row per **Category**, containing:
- `n_apps` — number of apps in that category
- `mean_rating` — average rating (rounded to 2 decimals)
- `mean_installs` — average installs (rounded to 0 decimals)
- `total_revenue` — sum of `Revenue`

Sort by `total_revenue` descending. Show only the top 10 rows.


In [21]:
# Groupby summary
summary = (
    df_dedup.groupby("Category")
    .agg(
        n_apps=("App", "count"),
        mean_rating=("Rating", "mean"),
        mean_installs=("Installs", "mean"),
        total_revenue=("Revenue", "sum")
    )
    .round(2)
    .sort_values(by="total_revenue", ascending=False)
    .head(10)
)

print(summary)

                 n_apps  mean_rating  mean_installs  total_revenue
Category                                                          
FAMILY             1718         4.19     5844662.74   1.857743e+08
LIFESTYLE           305         4.10     1753249.57   5.758394e+07
GAME               1074         4.28    29370449.46   4.098684e+07
FINANCE             317         4.13     2430007.57   2.572664e+07
PHOTOGRAPHY         304         4.18    31977773.45   8.941050e+06
MEDICAL             302         4.18      139611.51   8.371355e+06
PERSONALIZATION     310         4.33     6691461.06   7.786310e+06
TOOLS               734         4.05    15600442.10   5.462910e+06
SPORTS              286         4.23     5344515.61   4.706154e+06
PRODUCTIVITY        334         4.20    37314581.38   4.304452e+06


## Exercise 8 — Reflection (10 pts)

In 4–6 sentences, answer all of:

1. Which cleaning step would have been impossible to skip if you wanted to compute average install counts? Why?
2. What's one piece of information in the raw data that you *could* extract with more work but didn't (e.g., the `Size` column, the `Last Updated` column)? How would you approach it?
3. Did anything in the data surprise you?

YOUR ANSWER:
The cleaning step that would’ve been impossible to skip was converting the 'Installs' column into integers because the values had commas and plus signs in them, so pandas regarded the characters like strings. Without fixing that, you couldn’t actually calculate average installs or do comparisons. One thing that could still be extracted from the raw data is the 'Size' column since it uses different units like MB + KB. I’d probably clean it by removing letters and converting everything into one consistent unit so the sizes could be compared properly. One thing that surprised me was how messy a large dataset like this could be with duplicate apps, numeric columns, and missing string variables that are stored as strings.

## Submission checklist

- [ ] Name and date filled in
- [ ] All TODO cells completed and run
- [ ] All `YOUR ANSWER` prompts replaced
- [ ] **Restart & Run All** completes without errors
- [ ] Downloaded as `.ipynb` and uploaded to Canvas
